In [3]:
import sys
sys.path.append('../backend')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data_collection/training_data_verified.csv')

print(f"Shape: {df.shape}")
print(f"Unique users: {df['handle'].nunique()}")
print()
print(df.describe())

FileNotFoundError: [Errno 2] No such file or directory: '../data_collection/training_data_verified.csv'

In [3]:
import pandas as pd

# Convert tag features to dataframe for easy exploration
tag_data = []
for tag, features in profile['tag_features'].items():
    tag_data.append({
        'tag':            tag,
        'solve_rate':     features['solve_rate'],
        'avg_attempts':   features['avg_attempts'],
        'highest_rating': features['highest_rating'],
        'recency_days':   features['recency_days'],
        'total_solved':   features['total_solved']
    })

df = pd.DataFrame(tag_data)
df = df.sort_values('solve_rate', ascending=False)
print(df.to_string())

                        tag  solve_rate  avg_attempts  highest_rating  recency_days  total_solved
5                  *special       1.000          4.67            1200         724.4             3
9           graph matchings       1.000          1.67            1300         538.3             3
13            binary search       1.000          1.97            1700          12.3            39
11          dfs and similar       1.000          1.75            1600          17.4             8
12                   graphs       1.000          1.80            1400         163.3             5
10           shortest paths       1.000          1.00            1400         843.3             2
29                      fft       1.000          1.00            1700         483.1             1
23           ternary search       1.000          2.25            1400         632.3             4
22            probabilities       1.000          2.60            1600         416.8             5
20                  

In [4]:
print("Solve Rate Distribution:")
print(df['solve_rate'].describe())
print()
print("Highest Rating Distribution:")
print(df['highest_rating'].describe())
print()
print("Recency Distribution (days):")
print(df['recency_days'].describe())

Solve Rate Distribution:
count    31.000000
mean      0.973000
std       0.090003
min       0.500000
25%       0.981000
50%       1.000000
75%       1.000000
max       1.000000
Name: solve_rate, dtype: float64

Highest Rating Distribution:
count      31.000000
mean     1529.032258
std       234.084811
min       800.000000
25%      1400.000000
50%      1600.000000
75%      1700.000000
max      2000.000000
Name: highest_rating, dtype: float64

Recency Distribution (days):
count     31.000000
mean     275.361290
std      288.788815
min       12.300000
25%       14.900000
50%      163.300000
75%      501.850000
max      843.300000
Name: recency_days, dtype: float64


In [5]:
import numpy as np

def compute_skill_score(row, max_rating=3500):
    # Feature 1: solve rate (higher = better)
    solve_score = row['solve_rate']

    # Feature 2: attempt score (fewer attempts = better)
    # We invert so fewer attempts = higher score
    attempt_score = 1 / (1 + row['avg_attempts'])

    # Feature 3: rating score (higher problems solved = better)
    rating_score = row['highest_rating'] / max_rating

    # Feature 4: recency score (more recent = better)
    # Exponential decay: practiced yesterday = 1.0, 30 days ago = lower
    recency_score = np.exp(-row['recency_days'] / 30)

    # Weighted combination
    # Solve rate matters most (35%)
    # Rating level matters a lot (30%)
    # Recency matters (20%)
    # Attempts matter least (15%)
    score = (
        0.35 * solve_score   +
        0.30 * rating_score  +
        0.20 * recency_score +
        0.15 * attempt_score
    )
    return round(score, 4)

# Apply to all tags
df['skill_score'] = df.apply(compute_skill_score, axis=1)
df = df.sort_values('skill_score', ascending=True)

print("WEAKEST TOPICS:")
print(df.head(5)[['tag', 'skill_score', 'solve_rate', 'highest_rating']])
print()
print("STRONGEST TOPICS:")
print(df.tail(5)[['tag', 'skill_score', 'solve_rate', 'highest_rating']])

WEAKEST TOPICS:
                   tag  skill_score  solve_rate  highest_rating
26                 dsu       0.4648         0.5            1200
5             *special       0.4793         1.0            1200
25           schedules       0.4914         1.0            1400
19  expression parsing       0.4936         1.0             800
21             hashing       0.5119         1.0            1500

STRONGEST TOPICS:
                        tag  skill_score  solve_rate  highest_rating
1                    greedy       0.6580       0.971            1600
3               brute force       0.6710       0.991            1700
8   constructive algorithms       0.6714       0.979            1700
13            binary search       0.6789       1.000            1700
2                      math       0.6997       0.989            2000


In [8]:
def classify_skill(score):
    if score < 0.50:
        return "Weak"
    elif score < 0.60:
        return "Moderate"
    else:
        return "Strong"

df['level'] = df['skill_score'].apply(classify_skill)

print("Tag Distribution:")
print(df['level'].value_counts())
print()
print("Full breakdown:")
print(df[['tag', 'skill_score', 'level']].to_string())

Tag Distribution:
level
Moderate    14
Strong      13
Weak         4
Name: count, dtype: int64

Full breakdown:
                        tag  skill_score     level
26                      dsu       0.4648      Weak
5                  *special       0.4793      Weak
25                schedules       0.4914      Weak
19       expression parsing       0.4936      Weak
21                  hashing       0.5119  Moderate
23           ternary search       0.5162  Moderate
9           graph matchings       0.5176  Moderate
27                    trees       0.5193  Moderate
12                   graphs       0.5244  Moderate
22            probabilities       0.5288  Moderate
16                 geometry       0.5421  Moderate
10           shortest paths       0.5450  Moderate
15                 bitmasks       0.5480  Moderate
30              interactive       0.5621  Moderate
24            combinatorics       0.5633  Moderate
29                      fft       0.5707  Moderate
28                   